# Семинар_7 Методы ансамблирования моделей. Блендинг.

Цель семинара: освоить основные техники блендинга моделей

План семинара:

* Практика - Реализуем Hard и Soft voting с помощью voting classifier и без
* Практика - заблендим модели с разными весами
* Практика - сделаем бленд вашего решения с публичным решением
* Подведение итогов - проанализируем и обсудим результаты

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from sklearn import metrics

import numpy as np
from xgboost import XGBClassifier
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb

import pandas as pd
import time

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.10/site-packages/dask/dataframe/_pyarrow_compat.py:23: UserWarning: You are using pyarrow version 11.0.0 which is known to be insecure. See https://www.cve.org/CVERecord?id=CVE-2023-47248 for further details. Please upgrade to pyarrow>=14.0.1 or install pyarrow-hotfix to patch your current version.
  warnings.warn(


# 1. Soft Voting (15 мин).
Когда смешиваем вероятности

Сначала получим предсказания для моделей разными алгоритмами ml

Начнем с multiclass постановки задачи

In [2]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [3]:
target_col = ['Irrigation_Need']
drop_cols = ['id']
cat_cols = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type',
    'Water_Source', 'Mulching_Used', 'Region'
]
num_cols = [col for col in train.columns if col not in target_col + drop_cols + cat_cols]

In [4]:
X = train.drop(columns=target_col + drop_cols)
y = train['Irrigation_Need']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                  random_state=43, stratify=y)

In [6]:
X_train[cat_cols] = X_train[cat_cols].astype('category')
X_test[cat_cols] = X_test[cat_cols].astype('category')
test[cat_cols] = test[cat_cols].astype('category')

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [8]:
params = {
    'objective': 'multi:softmax',
    'enable_categorical': True,
    'tree_method': 'hist',
    'enable_categorical': True,
    'device': 'cuda',
    'seed': 42,
    'num_class': 3,
    'early_stopping_rounds' : 10,
}

xgb_model = XGBClassifier(**params)

xgb_model.fit(X_train, y_train,eval_set=[(X_test, y_test)],
                  verbose=50)

y_pred = xgb_model.predict(X_test)
balanced_accuracy_score(y_test, y_pred)

[0]	validation_0-mlogloss:0.72052
[50]	validation_0-mlogloss:0.05971
[99]	validation_0-mlogloss:0.05832


/opt/conda/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [16:44:52] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


0.9616524051676794

In [9]:
cb_mc = CatBoostClassifier(random_seed=43,
                           task_type='GPU',
                          cat_features=cat_cols, 
                           #learning_rate=0.1,
                           )

cb_mc.fit(X_train, y_train, eval_set=(X_test, y_test),
          verbose=100, early_stopping_rounds=100)

Learning rate set to 0.197141
0:	learn: 0.8767423	test: 0.8763533	best: 0.8763533 (0)	total: 6.98s	remaining: 1h 56m 17s
100:	learn: 0.0631795	test: 0.0650004	best: 0.0649996 (99)	total: 8.28s	remaining: 1m 13s
200:	learn: 0.0606420	test: 0.0636214	best: 0.0636214 (200)	total: 9.52s	remaining: 37.9s
300:	learn: 0.0586676	test: 0.0630519	best: 0.0630519 (300)	total: 10.7s	remaining: 25s
400:	learn: 0.0570449	test: 0.0626877	best: 0.0626877 (400)	total: 12s	remaining: 17.9s
500:	learn: 0.0554516	test: 0.0623793	best: 0.0623793 (500)	total: 13.2s	remaining: 13.2s
600:	learn: 0.0539744	test: 0.0621534	best: 0.0621395 (596)	total: 14.5s	remaining: 9.6s
700:	learn: 0.0526480	test: 0.0619946	best: 0.0619797 (696)	total: 15.7s	remaining: 6.69s
800:	learn: 0.0512844	test: 0.0619400	best: 0.0619400 (800)	total: 17s	remaining: 4.21s
900:	learn: 0.0501097	test: 0.0618860	best: 0.0618739 (888)	total: 18.2s	remaining: 2s
bestTest = 0.06187393431
bestIteration = 888
Shrink model to first 889 iteratio

In [10]:
y_pred = cb_mc.predict(X_test)
balanced_accuracy_score(y_test, y_pred)

0.9584328412824575

In [11]:
lg_mc = LGBMClassifier(random_state=43,
        objective="multiclass",
        n_estimators=500,
        num_class=3,
        verbose=-1,
        n_jobs=-1,
)

lg_mc.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(stopping_rounds=50)],
    )

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[65]	valid_0's multi_logloss: 0.0628849


LGBMClassifier(n_estimators=500, n_jobs=-1, num_class=3, objective='multiclass',
               random_state=43, verbose=-1)

In [12]:
y_pred = lg_mc.predict(X_test)
balanced_accuracy_score(y_test, y_pred)

0.9612435669259057

* Запустите [sklearn VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) (ссылка на доку) с нашими моделями 
* Измерьте roc_auc

In [13]:
lg_mc = LGBMClassifier(random_state=43,
        objective="multiclass",
        n_estimators=65,
        verbose=-1,
        num_class=3,
        n_jobs=-1,
)

params = {
    'objective': 'multi:softmax',
    'enable_categorical': True,
    'tree_method': 'hist',
    'enable_categorical': True,
    'device': 'cuda',
    'seed': 42,
    'num_class': 3,
}

xgb_mc = XGBClassifier(**params)

cb_mc = CatBoostClassifier(random_seed=43,
                           task_type='GPU',
                           cat_features=cat_cols, 
                           learning_rate=0.1971,
                           iterations=860,
                           verbose=0
                           )

models = [('xgb', xgb_mc), ('lgbm', lg_mc), ('cb', cb_mc)]

voting_soft = VotingClassifier(estimators=models, voting='soft')

voting_soft.fit(X_train, y_train)

y_pred_soft = voting_soft.predict(X_test)
balanced_accuracy_score(y_test, y_pred_soft)

0.9614031050506711

## Блендинг с весами
* Попробуйте добавить веса нашим моделям
* Подумайте, какой модели лучше присвоить наибольший вес, а какой наименьший 

In [14]:
weights = [0.4, 0.35, 0.25]
voting_soft = VotingClassifier(estimators=models, voting='soft', weights=weights)

voting_soft.fit(X_train, y_train)


y_pred_soft = voting_soft.predict(X_test)

balanced_accuracy_score(y_test, y_pred_soft)

0.9615892189043637

* Скор подрос - класс

# 2. Теперь попробуем сами реализовать hard voting (10 мин)
Выбираем самую встречаемую метку из всех предсказаний

Нам нужно предиктить не вероятности, а значение класса. 

In [15]:
params = {
    'objective': 'multi:softmax',
    'enable_categorical': True,
    'tree_method': 'hist',
    'enable_categorical': True,
    'device': 'cuda',
    'seed': 42,
    'num_class': 3,
    'early_stopping_rounds' : 10,
}

xgb_model = XGBClassifier(**params)

xgb_model.fit(X_train, y_train,eval_set=[(X_test, y_test)],
                  verbose=50)

xgb_pred_proba = xgb_model.predict_proba(X_test)
xgb_pred = xgb_model.predict(X_test)
balanced_accuracy_score(y_test, xgb_pred)

[0]	validation_0-mlogloss:0.72052
[50]	validation_0-mlogloss:0.05971
[99]	validation_0-mlogloss:0.05832


0.9616524051676794

In [16]:
cb_mc = CatBoostClassifier(random_seed=43,
                           task_type='GPU',
                          cat_features=cat_cols, 
                           #learning_rate=0.1,
                           )

cb_mc.fit(X_train, y_train, eval_set=(X_test, y_test),
          verbose=250, early_stopping_rounds=100)

Learning rate set to 0.197141
0:	learn: 0.8767423	test: 0.8763535	best: 0.8763535 (0)	total: 16.6ms	remaining: 16.6s
250:	learn: 0.0595676	test: 0.0633145	best: 0.0633145 (250)	total: 3.23s	remaining: 9.63s
500:	learn: 0.0553313	test: 0.0623347	best: 0.0623347 (500)	total: 6.46s	remaining: 6.44s
750:	learn: 0.0519427	test: 0.0620781	best: 0.0620670 (742)	total: 9.66s	remaining: 3.2s
999:	learn: 0.0489772	test: 0.0620053	best: 0.0619858 (942)	total: 12.9s	remaining: 0us
bestTest = 0.06198579334
bestIteration = 942
Shrink model to first 943 iterations.


In [17]:
cb_pred_proba = cb_mc.predict_proba(X_test)
cb_pred = cb_mc.predict(X_test)
balanced_accuracy_score(y_test, cb_pred)

0.957943537924483

In [18]:
lg_mc = LGBMClassifier(random_state=43,
        objective="multiclass",
        n_estimators=500,
        num_class=3,
        verbose=-1,
        n_jobs=-1,
)

lg_mc.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(stopping_rounds=50)],
    )

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[65]	valid_0's multi_logloss: 0.0628849


LGBMClassifier(n_estimators=500, n_jobs=-1, num_class=3, objective='multiclass',
               random_state=43, verbose=-1)

In [19]:
lg_pred_proba = lg_mc.predict_proba(X_test)
lg_pred = lg_mc.predict(X_test)
balanced_accuracy_score(y_test, lg_pred)

0.9612435669259057

Мы получили hard labels предсказания. Они хранятся в переменных `cb_pred`, `lg_pred`, `xgb_pred`.

Сделайте бленд решением большинства, например, с помощью [функции `mode`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mode.html) из `scipy` или другим способом.

In [20]:
from scipy.stats import mode

predictions = np.vstack((xgb_pred, cb_pred.ravel(), lg_pred))
predictions

array([[1, 1, 1, ..., 1, 1, 2],
       [1, 1, 1, ..., 1, 1, 2],
       [1, 1, 1, ..., 1, 1, 2]])

In [21]:
hard_voting_predictions, _ = mode(predictions, axis=0)

In [22]:
balanced_accuracy_score(y_test, hard_voting_predictions)

0.9610839331542206

# 4. Бленд с весами. Оптимизация значений весов (10 мин).

* Мы можем добавить веса каждому из пресказаний бленда
* Подберем оптюной значения весов

In [23]:
import optuna

optuna.logging.set_verbosity(optuna.logging.ERROR)
def objective(trial):

    a = trial.suggest_float('a', 0, 1)
    b = trial.suggest_float('b', 0, 1 - a)
    c = 1 - a - b
    
    voting_predictions = a * xgb_pred_proba + b * lg_pred_proba + c * cb_pred_proba
    voting_predictions = np.argmax(voting_predictions, axis=1)
    
    score = balanced_accuracy_score(y_test, voting_predictions)
    
    return score

In [24]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)

best_weights = study.best_params
best_weights

/opt/conda/lib/python3.10/site-packages/optuna/samplers/_tpe/sampler.py:529: RuntimeWarning: invalid value encountered in subtract
  acq_func_vals = log_likelihoods_below - log_likelihoods_above
/opt/conda/lib/python3.10/site-packages/optuna/samplers/_tpe/sampler.py:529: RuntimeWarning: invalid value encountered in subtract
  acq_func_vals = log_likelihoods_below - log_likelihoods_above
/opt/conda/lib/python3.10/site-packages/optuna/samplers/_tpe/sampler.py:529: RuntimeWarning: invalid value encountered in subtract
  acq_func_vals = log_likelihoods_below - log_likelihoods_above
/opt/conda/lib/python3.10/site-packages/optuna/samplers/_tpe/sampler.py:529: RuntimeWarning: invalid value encountered in subtract
  acq_func_vals = log_likelihoods_below - log_likelihoods_above
/opt/conda/lib/python3.10/site-packages/optuna/samplers/_tpe/sampler.py:529: RuntimeWarning: invalid value encountered in subtract
  acq_func_vals = log_likelihoods_below - log_likelihoods_above
/opt/conda/lib/python3.10

{'a': 0.2556366904761999, 'b': 0.6079628872756321}

In [25]:
best_a = best_weights['a']
best_b = best_weights['b']
best_c = 1 - best_a - best_b

best_voting_predictions = best_a * xgb_pred_proba + best_b * lg_pred_proba + best_c * cb_pred_proba
best_voting_predictions = np.argmax(best_voting_predictions, axis=1)

balanced_accuracy_score(y_test, best_voting_predictions)

0.962024900233502

* Выведите на экран, какие веса мы подобрали

In [26]:
print(best_a, best_b, best_c)

0.2556366904761999 0.6079628872756321 0.13640042224816795


# 5. Бленд с публичным решением.

* Получите предсказания на тестовых данных разными мл моделями. Сделайте бленд.
* Скачайте предсказания из публичнном ноуте. Например [из этого](https://www.kaggle.com/code/khsamaha/spd-multi-label-xgboost-mlr-r/output). Сблендите с вашими предсказаниями. 
* *Отошлите сабмишн на лб.

In [27]:
cb_preds = cb_mc.predict_proba(test.drop('id', axis=1))
xgb_preds = xgb_model.predict_proba(test.drop('id', axis=1))
lg_preds = lg_mc.predict_proba(test.drop('id', axis=1))

# best_a, best_b - наши веса
predicts = best_a * xgb_preds + best_b * lg_preds + best_c * cb_preds

predicts = np.argmax(predicts, axis=1)
predicts = le.inverse_transform(predicts)

Добавьте предикты из ноутбука по ссылке.

Для этого нажмите add input и сделайте поиск по названию ноутбука.

Получите предсказания из ноутбуков и сблендите с нашими.

In [28]:
sub2 = pd.read_csv('/kaggle/input/notebooks/amanatar/ps6e4-kgmon-playbook-grandmaster-stack-try/submission.csv')
sub3 = pd.read_csv('/kaggle/input/notebooks/mohit78241/s6e4-ensemble-voting-transfer-0-981-lb/submission.csv')
sub4 = pd.read_csv('/kaggle/input/notebooks/ravi20076/playgrounds6e4-tunedblend-v1/submission.csv')
#sub5 = ###

Добавте к нашему бленду еще один или несколько [публичных ноутбуков](https://www.kaggle.com/competitions/playground-series-s6e4/code) с предсказаниями (можно поставить фильтр Public score)

Старайтесь выбирать для бленда модели, которые не обучали сами.

In [29]:
predictions = np.vstack((predicts, sub2['Irrigation_Need'], sub3['Irrigation_Need'], sub4['Irrigation_Need']))
df = pd.DataFrame(predictions.T)  

preds_fin = df.mode(axis=1)[0].values

In [30]:
sub2['Irrigation_Need'] = preds_fin

In [31]:
sub2.to_csv('submission.csv', index=False)

## 